# Notebook 05 — Pledge Quality Score (PQS)

**Project:** Corporate Climate Credibility Audit — power-sector POC

**Author:** Souvik Mandal (drsouvikmandal@gmail.com)

**Notebook role:** Score each cohort parent's **public climate pledge quality** across five structured criteria. 
- "Ambition": Magnitude of reduction commitment + pathway alignment + scope coverage.
- "Specificity": Concreteness of target definition — base year, target year, unambiguous numbers.
- "Time horizon": Presence of near-term + long-term targets with interim milestones.
- "Mechanism clarity": Specificity of HOW the company will achieve its target — naming concrete actions, capex, and timing.
- "Verification governance": Independent validation + accountability infrastructure.

For the detailed structure of each parameter and the scoring method, please see the "SCORING RUBRIC (CRITERIA + POINTS) FOR LLM EVALUATION" section below. The rubric is centralized in `scoring_prompt` for transparency and easy modification.

This is the **Talk axis** of the CCCS framework (Talk × Walk × Verify). Output `cohort_pqs.csv` is consumed by Notebook 06 (CCCS Composite) as the Talk-axis input.

## Prerequisites — API keys

This notebook calls the **Anthropic** and **OpenAI** Chat APIs to score each parent's disclosure corpus on the five rubric criteria. Both providers are required (the inter-model agreement statistic is the only reproducibility check the notebook produces, so a single-model run is not a valid substitute).

To obtain API keys:

- **Anthropic** — create an account at <https://console.anthropic.com/> and issue a key under *Settings → API Keys*: <https://console.anthropic.com/settings/keys>
- **OpenAI** — create an account at <https://platform.openai.com/> and issue a key under *API Keys*: <https://platform.openai.com/api-keys>

Save both keys in the project-root `.env` file (same directory as this notebook's parent `/notebooks/` folder), using **exactly** these variable names:

```env
ANTHROPIC_API_KEY=sk-ant-...
OPENAI_API_KEY=sk-...
```

The environment cell (Section 0) loads `.env` via `python-dotenv` and prints a presence-check for both keys before any API call is issued. Approximate end-to-end API cost for the full cohort (~26 scored parents × 2 models) is on the order of single-digit US dollars; per-call cost is printed inline in Sections 4 and 5.

## SCORING RUBRIC (CRITERIA + POINTS) FOR LLM EVALUATION

In [1]:
# Define the independent scoring rubric for the LLM evaluation, with 5 criteria each scored 0-10 based on specific criteria. This will be used to guide the LLM's assessment of each company's disclosure quality and ambition.

scoring_prompt = """

CRITERION 1 — AMBITION (0-10)
Magnitude of reduction commitment + pathway alignment + scope coverage.
  0  = No public emissions-reduction commitment found in the disclosure.
  2  = Vague aspiration ("committed to reducing emissions"; no number; no year).
  4  = General net-zero or carbon-neutral pledge with target year (e.g., "net-zero by 2050") but no specific reduction percentage AND no explicit pathway alignment.
  6  = Specific reduction percentage stated (e.g., "50% reduction by 2030") but pathway alignment unclear OR scope limited to Scope 1 only.
  8  = Specific reduction with explicit pathway alignment (1.5°C, well-below-2°C, or "science-based") covering at least Scope 1+2.
  10 = Specific reduction explicitly aligned with 1.5°C pathway, covering Scope 1+2+3 OR net-zero across all scopes, independently validated (SBTi or equivalent third party).

CRITERION 2 — SPECIFICITY (0-10)
Concreteness of target definition — base year, target year, unambiguous numbers.
  0  = No defined base year AND no defined target year.
  2  = Either base year OR target year defined (not both).
  4  = Both base year and target year defined, but uses hedging language ("approximately", "around", "up to").
  6  = Both years defined, specific reduction value, single end target only.
  8  = Multiple targets with distinct base years and target years, all numerically precise.
  10 = Comprehensive disclosure: base year + multiple interim targets each with explicit year + end target, with measurable annual or quinquennial milestones.

CRITERION 3 — TIME HORIZON STRUCTURE (0-10)
Presence of near-term + long-term targets with interim milestones.
  0  = No targets at all, OR only one target with no time horizon.
  2  = Single long-term-only target (e.g., 2050 net-zero, nothing earlier).
  4  = Long-term target + one near-term commitment (within 5 years of disclosure date).
  6  = Long-term target + at least one interim target between 5-15 years out.
  8  = Long-term target + multiple interim targets (e.g., 2030, 2040, 2050).
  10 = Long-term + at least two interim targets + explicit annual/quinquennial milestones with progress tracking disclosed.

CRITERION 4 — MECHANISM CLARITY (0-10)
Specificity of HOW the company will achieve its target — naming concrete actions, capex, and timing.
  0  = No mention of how decarbonization will be achieved.
  2  = Vague pathways only ("we will continue to invest in clean energy"; "we will reduce emissions through efficiency").
  4  = Lists 1-2 specific mechanisms (e.g., coal-to-gas fuel switching, REC procurement) without quantification.
  6  = Multiple specific mechanisms (fuel switching, electrification, CCS, renewables build-out) with at least one quantified ("invest $X billion in renewable capacity").
  8  = Comprehensive mechanism portfolio with capex commitments tied to specific units, dates, AND clear treatment of offsets (e.g., "no more than 10% of reductions from offsets").
  10 = Detailed transition plan with named projects, specific dates, capex amounts, and explicit offset disclosure with quantified limits.

CRITERION 5 — VERIFICATION & GOVERNANCE (0-10)
Independent validation + accountability infrastructure.
  Award 2 points for EACH of the following four elements present (cap at 10):
   • External target validation: SBTi-validated targets, third-party scientific review, or peer assurance organization.
   • Third-party emissions assurance: independent auditor (Big-4, ERM, etc.) attests annual emissions reporting.
   • Board-level climate oversight: dedicated board committee or named director with climate mandate.
   • Executive compensation linkage: portion of CEO or executive pay explicitly tied to climate metrics.
   Plus 2 bonus points if all four are present AND annual public progress reporting is disclosed.

"""

## Methodology

**Why not SBTi alone?** SBTi targets data is the most rigorous and harmonized public corpus of corporate climate targets, so it is the natural first candidate to source the 5-criterion text. However, an initial coverage probe (run programmatically in Section 1) finds **only 2 of 50 cohort parents have SBTi entries** (Vistra, NRG). This is itself a finding — SBTi adoption in the U.S. power sector is sparse — but it means SBTi alone is insufficient as a disclosure source for this cohort. The notebook therefore broadens the source set.

**Disclosure-source design.** Three sources per parent (combined when present):

- **SEC 10-K** Items 1A (Risk Factors) + 7 (MD&A), filtered to climate-relevant paragraphs, for cohort parents that file with the SEC.
- **Net Zero Tracker (NZT)** structured fields (end target, interim target, scopes covered, published transition plan, governance, source URL), for cohort parents NZT covers. NZT-to-cohort matching is hand-curated in Section 1 to filter out false positives that arise from token-overlap matching (e.g., "American Express" colliding with "American Electric Power").
- **SBTi target language** for the parents that do have SBTi entries.

**Why LLM scoring rather than regex / keyword density?** The 5 criteria (Ambition, Specificity, Time Horizon, Mechanism Clarity, Verification & Governance) reward concreteness that text-density cannot capture. Hedging language ("approximately", "we may consider"), conditional commitments, and the difference between "$2B by 2028 to convert coal to gas" vs. "we will continue to invest in clean energy" are exactly the distinctions a keyword counter conflates. An LLM at `temperature=0` produces deterministic scores, reasoning, and verbatim evidence quotes that a human reviewer can audit. The output cost is small (reported per call in Section 4's dry-run printout).

**Why two LLMs?** Single-model scoring is a black box: we cannot tell whether a score reflects the disclosure or the model's idiosyncrasies. Scoring with two independent frontier models (Claude Sonnet 4.6 + GPT-4o, same system prompt, same `temperature=0`) and averaging criterion-wise gives us (i) noise reduction and (ii) a directly reportable reproducibility statistic — per-criterion Pearson r and mean-absolute-difference between models — which Section 8 saves as `cohort_pqs_intermodel_agreement.csv`.

## Coverage projection

| Source | Cohort coverage |
|---|---:|
| SEC 10-K (registrants) | 22/50 |
| Net Zero Tracker | 23/50 |
| SBTi | 2/50 |
| **Union (any disclosure)** | **26/50** |
| Talk-NA flag (cooperatives + public-power + PE) | 24/50 |

The 24 Talk-NA parents are cooperatives, public-power authorities, PE-owned holdcos, and government entities (e.g., TVA) for which no public disclosure source we can access exists. They inherit `pqs_composite = NaN` and are documented in the working paper as a known limitation of the Talk axis.

## Data Schema
### Inputs

| File | Producer | Used here |
|---|---|---|
| `data/processed/cohort_top50.csv` | Notebook 00 | Cohort list |
| `data/processed/parent_to_cik.csv` | Notebook 00 | Find 10-K filings |
| `data/processed/tenk_filings_index.csv` | Notebook 00 | 10-K file paths |
| `data/raw/sec_10k/{parent}/*.htm` | Notebook 00 | 10-K text |
| `data/processed/nzt_us_companies.csv` | Notebook 00 §7 | NZT structured fields |
| `data/raw/sbti/targets-excel.xlsx` | Notebook 00 §5 | SBTi target language |
| `.env` → `ANTHROPIC_API_KEY` + `OPENAI_API_KEY` | user | LLM calls |

### Outputs

| File | Description |
|---|---|
| `data/processed/cohort_pqs.csv` | Per-parent composite PQS (0-100) + per-criterion scores (0-10) — the deliverable for Notebook 06 |
| `data/processed/cohort_pqs_intermodel_agreement.csv` | Per-criterion Pearson correlation and MAD between Claude and OpenAI |
| `data/processed/cohort_pqs_llm_reasoning.json` | Full per-parent per-model reasoning + evidence quotes for audit |
| `data/processed/scratch/disclosure_corpus/*.txt` | Per-parent disclosure corpora (intermediate; reproducibility) |
| `data/processed/scratch/llm_scores/*.json` | Per-parent per-model raw JSON outputs (cached) |

---

## Section map

0. Environment + helpers
1. Load disclosure sources (10-K index, NZT, SBTi)
2. Build per-parent disclosure corpora (idempotent — skips parents already on disk)
3. LLM scoring framework (system prompt + API helpers + cost estimator)
4. **Dry-run** on 5 representative parents → review before bulk
5. **Bulk** scoring on all 26 parents with disclosure (idempotent)
6. Composite PQS construction (dual-model average + 5-criterion equal-weight)
7. Sensitivity analysis on criterion weights
8. Save deliverables
9. Summary of results

## 0. Environment

In [2]:
from __future__ import annotations
import os, re, json, time, warnings
from pathlib import Path
from typing import Optional
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# .env loading (mirrors all other notebooks)
try:
    from dotenv import load_dotenv
    _ROOT = Path("/Users/souvikmandal/Documents/S00_Career-development/20260318_HBS_Sen-Data-Scientist/task")
    for _p in [_ROOT / ".env", Path.cwd() / ".env", Path.cwd().parent / ".env"]:
        if _p.exists(): load_dotenv(_p, override=False); break
except ImportError:
    pass

PROJECT_ROOT   = Path("/Users/souvikmandal/Documents/S00_Career-development/20260318_HBS_Sen-Data-Scientist/task")
DATA_RAW       = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
SCRATCH_CORPUS = DATA_PROCESSED / "scratch" / "disclosure_corpus"
SCRATCH_SCORES = DATA_PROCESSED / "scratch" / "llm_scores"
FIG_DIR        = PROJECT_ROOT / "outputs" / "figures"
for d in (SCRATCH_CORPUS, SCRATCH_SCORES, FIG_DIR): d.mkdir(parents=True, exist_ok=True)

# API key sanity check
ANTHROPIC_KEY = os.environ.get("ANTHROPIC_API_KEY", "").strip()
OPENAI_KEY    = os.environ.get("OPENAI_API_KEY",    "").strip()
print(f"ANTHROPIC_API_KEY present: {bool(ANTHROPIC_KEY)} (len {len(ANTHROPIC_KEY)})")
print(f"OPENAI_API_KEY    present: {bool(OPENAI_KEY)} (len {len(OPENAI_KEY)})")

warnings.filterwarnings("ignore")
RNG_SEED = 20260524
np.random.seed(RNG_SEED)


def safe_name(s: str) -> str:
    return re.sub(r"[^A-Za-z0-9]+", "_", str(s)).strip("_")

ANTHROPIC_API_KEY present: True (len 108)
OPENAI_API_KEY    present: True (len 164)


## 1. Load disclosure sources

Three inputs, all already on disk from Notebook 00 acquisition. We build a coverage matrix to know which parents have which sources.

In [3]:
# --- cohort + 10-K index + NZT + SBTi targets ---
cohort       = pd.read_csv(DATA_PROCESSED / "cohort_top50.csv")
tenk_index   = pd.read_csv(DATA_PROCESSED / "tenk_filings_index.csv")
nzt          = pd.read_csv(DATA_PROCESSED / "nzt_us_companies.csv")
sbti_targets = pd.read_excel(DATA_RAW / "sbti" / "targets-excel.xlsx")
sbti_targets.columns = [str(c).strip() for c in sbti_targets.columns]

print(f"Cohort:           {len(cohort)} parents")
print(f"10-K filings:     {len(tenk_index)} parents indexed")
print(f"NZT US companies: {len(nzt)} entities")
print(f"SBTi targets:     {len(sbti_targets):,} target rows globally")

Cohort:           50 parents
10-K filings:     25 parents indexed
NZT US companies: 686 entities
SBTi targets:     37,813 target rows globally


In [4]:
# --- TRUE-POSITIVE NZT matches (hand-curated; filters out false positives like
# American Express → AEP that come from the 'American' token overlap) ---
NZT_MANUAL_MATCHES = {
    "Vistra Corp":                              "Vistra Corp.",
    "THE SOUTHERN CO":                          "Southern Company",
    "DUKE ENERGY CORP":                         "Duke Energy",
    "BERKSHIRE HATHAWAY INC":                   "Berkshire Hathaway",
    "AMERICAN ELECTRIC POWER CO INC":           "American Electric Power Company",
    "NEXTERA ENERGY INC":                       "NextEra Energy",
    "ENTERGY CORP":                             "Entergy",
    "XCEL ENERGY INC":                          "Xcel Energy",
    "DOMINION ENERGY INC":                      "Dominion Energy",
    "PPL CORP":                                 "PPL",
    "EVERGY INC":                               "Evergy",
    "DTE ENERGY CO":                            "DTE Energy",
    "NRG ENERGY INC":                           "NRG Energy",
    "AMEREN CORP":                              "Ameren",
    "WEC Energy Group Inc":                     "WEC Energy",
    "CMS ENERGY CORP":                          "CMS Energy",
    "FIRSTENERGY CORP":                         "FirstEnergy",
    "ALLIANT ENERGY CORP":                      "Alliant Energy",
    "AES CORP":                                 "AES",
    "PINNACLE WEST CAPITAL CORP":               "Pinnacle West Capital",
    "CONSTELLATION ENERGY CORP":                "Constellation Energy",
    "MARATHON PETROLEUM CORP":                  "Marathon Petroleum",
    "EXXON MOBIL CORP":                         "Exxon Mobil",
}
print(f"True-positive NZT matches for cohort: {len(NZT_MANUAL_MATCHES)}")

True-positive NZT matches for cohort: 23


In [5]:
# --- Build coverage matrix ---
coverage = []
for _, c in cohort.iterrows():
    parent = c["parent_name"]
    has_10k  = (DATA_RAW / "sec_10k" / parent).exists() and any((DATA_RAW / "sec_10k" / parent).glob("*.htm*"))
    has_nzt  = parent in NZT_MANUAL_MATCHES
    has_sbti = sbti_targets["company_name"].astype(str).str.lower().str.contains(parent.lower()[:20], na=False).any()
    coverage.append({"parent_name": parent, "has_10k": has_10k, "has_nzt": has_nzt, "has_sbti": has_sbti,
                      "n_sources": int(has_10k) + int(has_nzt) + int(has_sbti)})
coverage_df = pd.DataFrame(coverage)
print(f"\\nCoverage summary:")
print(f"  Parents with any disclosure source: {(coverage_df['n_sources'] > 0).sum()} / {len(coverage_df)}")
print(f"  with 10-K:  {coverage_df['has_10k'].sum()}")
print(f"  with NZT:   {coverage_df['has_nzt'].sum()}")
print(f"  with SBTi:  {coverage_df['has_sbti'].sum()}")
print(f"  Talk-NA (no source): {(coverage_df['n_sources'] == 0).sum()}")

\nCoverage summary:
  Parents with any disclosure source: 26 / 50
  with 10-K:  26
  with NZT:   23
  with SBTi:  2
  Talk-NA (no source): 24


## 2. Build per-parent disclosure corpora

For each cohort parent with at least one available source, build a single text corpus with `[SOURCE: X]` labels separating the SEC 10-K, NZT, and SBTi blocks. The labels matter: at scoring time the LLM is asked to attribute evidence to a source, and the labels make that attribution unambiguous.

Two design choices are worth flagging:

- **Climate-keyword filtering on 10-K text.** Items 1A + 7 of a 10-K average tens of thousands of words; most of that is non-climate (operating segments, legal proceedings, liquidity, etc.). We keep only paragraphs matching a climate keyword regex (plus ±1 neighbor paragraph for context). This keeps the corpus focused on what the rubric actually scores and bounds API cost.
- **Idempotent caching.** Each per-parent corpus is written to `data/processed/scratch/disclosure_corpus/{parent}.txt` and re-used on subsequent runs unless `force_rebuild=True` is passed. This makes prompt iteration cheap: tweaking the LLM rubric does not require re-parsing 22 10-Ks.

Corpus token counts (approximate, 4 chars ≈ 1 token) are printed at the end of this section so the reader can see the size distribution before any LLM is called.

In [6]:
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

CLIMATE_KEYWORDS = re.compile(
    r"\\b(climate|carbon|emission|GHG|greenhouse|decarbon|net.?zero|renewable|"
    r"sustainabilit|ESG|scope\\s+[123]|mitigation|coal\\s+retire|gas\\s+transition|"
    r"electrif|CCS|carbon\\s+captur|methane|biomass|net.?carbon|carbon\\s+neutral)",
    re.IGNORECASE
)


def extract_section(text: str, start_pat: str, end_pat: str, max_chars: int = 200_000) -> str:
    """Pull the LAST occurrence of start_pat (skipping the TOC entry) up to end_pat."""
    matches = [(m.start(), m.end()) for m in re.finditer(start_pat, text, re.IGNORECASE)]
    if not matches: return ""
    start = matches[-1][1]
    end_matches = [m.start() for m in re.finditer(end_pat, text[start:], re.IGNORECASE)]
    end = start + end_matches[0] if end_matches else start + max_chars
    return text[start:min(end, start + max_chars)]


def climate_paragraphs(section_text: str, context: int = 1) -> str:
    """Return paragraphs containing climate keywords plus ±context neighbors."""
    paras = [p.strip() for p in re.split(r"\\n\\s*\\n", section_text) if len(p.strip()) > 50]
    keep = set()
    for i, p in enumerate(paras):
        if CLIMATE_KEYWORDS.search(p):
            keep.add(i)
            for k in range(1, context + 1):
                keep.add(i - k); keep.add(i + k)
    keep = sorted(i for i in keep if 0 <= i < len(paras))
    return "\\n\\n".join(paras[i] for i in keep)


def parse_10k(html_path: Path) -> tuple[str, str]:
    """Return (item_1a_climate_text, item_7_climate_text)."""
    html = html_path.read_text(errors="replace")
    soup = BeautifulSoup(html, "lxml")
    text = soup.get_text(separator="\\n")
    item_1a = extract_section(text, r"item\\s+1a\\.?\\s*risk\\s+factors", r"item\\s+1b\\.?\\s*unresolved")
    item_7  = extract_section(text, r"item\\s+7\\.?\\s*management.?s\\s+discussion", r"item\\s+7a\\.?\\s*quantitative")
    return climate_paragraphs(item_1a), climate_paragraphs(item_7)


def nzt_block_for(parent: str) -> Optional[str]:
    if parent not in NZT_MANUAL_MATCHES: return None
    sub = nzt[nzt["Name"] == NZT_MANUAL_MATCHES[parent]]
    if len(sub) == 0: return None
    d = sub.iloc[0]
    return (
        f"Entity: {d['Name']}\\n"
        f"Industry: {d['Industry']}\\n"
        f"End target: {d['End_target']}  (year {d['End_target_year']}; % {d['End_target_percentage_reduction']}; baseline {d['End_target_baseline_year']})\\n"
        f"  Status: {d['Status_of_end_target']}\\n"
        f"  Description: {str(d['End_target_text'])[:600]}\\n"
        f"Interim target: {d['Interim_target']}  (year {d['Interim_target_year']}; % {d['Interim_target_percentage_reduction']})\\n"
        f"  Description: {str(d['Interim_target_text'])[:400]}\\n"
        f"Published transition plan: {d['Published_plan']}\\n"
        f"Reporting mechanism: {d['Reporting_mechanism']}\\n"
        f"Accountability/governance: {d['Accountability_delivery']}\\n"
        f"Race to Zero member: {d['Race_to_zero_member']}\\n"
        f"Scopes covered: 1={d['Scope_1_coverage']}, 2={d['Scope_2_coverage']}, 3={d['Scope_3_coverage']}\\n"
        f"Source URL: {d['End_target_source_url']}"
    )


def sbti_block_for(parent: str) -> Optional[str]:
    norm = parent.lower()[:20]
    hits = sbti_targets[sbti_targets["company_name"].astype(str).str.lower().str.contains(norm, na=False)]
    if len(hits) == 0: return None
    lines = []
    for _, r in hits.iterrows():
        lines.append(f"action={r.get('action')}, status={r.get('status')}, scope={r.get('scope')}, target_year={r.get('target_year')}, target_value={r.get('target_value')}, classification={r.get('target_classification_short')}")
        lang = r.get("full_target_language")
        if pd.notna(lang): lines.append(f"  full_target_language: {str(lang)[:500]}")
    return "\\n".join(lines)


def build_corpus(parent: str, force_rebuild: bool = False) -> Optional[str]:
    """Return corpus text for a parent (or None if no sources). Cached to scratch."""
    out_path = SCRATCH_CORPUS / f"{safe_name(parent)}.txt"
    if out_path.exists() and not force_rebuild:
        return out_path.read_text()

    parts = []
    tenk_dir = DATA_RAW / "sec_10k" / parent
    if tenk_dir.exists():
        htm_files = list(tenk_dir.glob("*.htm")) + list(tenk_dir.glob("*.html"))
        if htm_files:
            try:
                ia, i7 = parse_10k(htm_files[0])
                parts.append(f"[SOURCE: SEC 10-K Item 1A — Risk Factors (climate-relevant excerpts)]\\n\\n{ia}")
                parts.append(f"[SOURCE: SEC 10-K Item 7 — MD&A (climate-relevant excerpts)]\\n\\n{i7}")
            except Exception as e:
                parts.append(f"[SOURCE: SEC 10-K — PARSE_ERROR: {e}]")
    nzt_text = nzt_block_for(parent)
    if nzt_text:
        parts.append(f"[SOURCE: Net Zero Tracker]\\n\\n{nzt_text}")
    sbti_text = sbti_block_for(parent)
    if sbti_text:
        parts.append(f"[SOURCE: Science Based Targets initiative (SBTi)]\\n\\n{sbti_text}")

    if not parts:
        return None
    corpus = "\\n\\n".join(parts)
    out_path.write_text(corpus)
    return corpus

In [7]:
# Build corpora for all cohort parents (idempotent)
from tqdm.auto import tqdm
stats = []
for _, c in tqdm(cohort.iterrows(), total=len(cohort), desc="Building corpora"):
    parent = c["parent_name"]
    corpus = build_corpus(parent)
    stats.append({"parent": parent, "has_corpus": corpus is not None,
                   "n_chars": len(corpus) if corpus else 0,
                   "n_tokens_approx": len(corpus) // 4 if corpus else 0})

corpus_stats = pd.DataFrame(stats).sort_values("n_tokens_approx", ascending=False)
print(f"\\nParents with disclosure corpus: {(corpus_stats['has_corpus']).sum()} / {len(corpus_stats)}")
print(f"Token-count distribution (approx):")
print(corpus_stats[corpus_stats['has_corpus']]['n_tokens_approx'].describe().round(0).to_string())
corpus_stats.to_csv(DATA_PROCESSED / "scratch" / "disclosure_corpus_stats.csv", index=False)

Building corpora: 100%|██████████| 50/50 [00:00<00:00, 246.97it/s]

\nParents with disclosure corpus: 26 / 50
Token-count distribution (approx):
count       26.0
mean     24628.0
std      18372.0
min        285.0
25%       9829.0
50%      23220.0
75%      36798.0
max      62739.0


## 3. LLM scoring framework

The system prompt (defined in the next code cell) combines two pieces:

1. A fixed structural template that pins the response format to strict JSON with per-criterion `score` + `reasoning` + `evidence_quote`, plus a supplemental `sbti_status` indicator and `data_quality_flags`.
2. The 5-criterion rubric defined at the top of the notebook, injected verbatim.

Both Claude Sonnet 4.6 and GPT-4o are called with the same system prompt, the same user template, and `temperature=0` for determinism. Per-call token usage and dollar cost are returned alongside each scoring result, so the dry-run cell in Section 4 prints actual cost rather than relying on a pre-computed estimate.

Two pragmatic guards in the code cell below:

- `_compact_corpus()` truncates very long corpora (head + tail, with an inline note) so a single call stays under organization-level token-per-minute ceilings. The threshold is exposed as `MAX_LLM_CORPUS_CHARS` for tuning.
- `_extract_json_object()` parses the model response defensively: it strips markdown fences, locates the first/last brace, and only then `json.loads` — this absorbs the occasional model preamble without losing the structured score.

In [8]:
# ============================================
# THE CORE SYSTEM PROMPT STRUCTURAL TEMPLATES
# ============================================
# Define the core structural prompt template with a clean placeholder

SYSTEM_PROMPT_TEMPLATE = """You are an experienced ESG analyst scoring U.S. power-sector companies on the quality of their public climate-emissions pledge. You will see a "disclosure corpus" labeled with [SOURCE: X] markers (SEC 10-K, Net Zero Tracker, SBTi). Score each of 5 criteria 0-10. Be conservative. Defend each score with brief reasoning and (where possible) a verbatim quote.

### SCORING RUBRIC
[SCORING_RUBRIC]

OUTPUT — strict JSON only:
{
  "parent_name": "<as given>",
  "overall_assessment": "<1-2 sentences>",
  "criteria": {
    "ambition":                {"score": <int 0-10 or null>, "reasoning": "...", "evidence_quote": "..."},
    "specificity":             {"score": <int 0-10 or null>, "reasoning": "...", "evidence_quote": "..."},
    "time_horizon":            {"score": <int 0-10 or null>, "reasoning": "...", "evidence_quote": "..."},
    "mechanism_clarity":       {"score": <int 0-10 or null>, "reasoning": "...", "evidence_quote": "..."},
    "verification_governance": {"score": <int 0-10 or null>, "reasoning": "...", "evidence_quote": "..."}
  },
  "supplemental_indicators": {"sbti_status": "validated_targets | targets_set | committed | removed | not_present | unknown"},
  "data_quality_flags": ["no_10k_text", "no_nzt_entry", "no_sbti_entry", "sparse_disclosure", "conflicting_information"],
  "confidence": "high | medium | low"
}

Use null (not 0) when information is genuinely missing; use 0 only when there IS disclosure but the company makes no commitment on that dimension."""


# 3. Combine them programmatically at runtime
def generate_system_prompt(rubric: str) -> str:
    return SYSTEM_PROMPT_TEMPLATE.replace("[SCORING_RUBRIC]", rubric.strip())

# 4. Compile the final prompt for your API call
SYSTEM_PROMPT = generate_system_prompt(scoring_prompt)

In [9]:
USER_TEMPLATE = """Parent company: {parent_name}

Disclosure corpus follows. Use only this text; do not draw on outside knowledge.

<<<DISCLOSURE_CORPUS_START>>>
{corpus}
<<<DISCLOSURE_CORPUS_END>>>

Score per the rubric. Return only the JSON."""

In [10]:
# --- API client helpers (with criteria-schema enforcement) ---
# Install on first use
try: import anthropic
except ImportError:
    import subprocess; subprocess.check_call(["pip", "install", "-q", "anthropic"]); import anthropic
try: import openai
except ImportError:
    import subprocess; subprocess.check_call(["pip", "install", "-q", "openai"]); import openai

CLAUDE_MODEL = "claude-sonnet-4-6"
OPENAI_MODEL = "gpt-4o"

claude_client = anthropic.Anthropic(api_key=ANTHROPIC_KEY) if ANTHROPIC_KEY else None
openai_client = openai.OpenAI(api_key=OPENAI_KEY) if OPENAI_KEY else None

# Cost per 1K tokens (approximate; check current pricing).
COSTS = {
    "claude_in":  0.003,  "claude_out": 0.015,
    "openai_in":  0.0025, "openai_out": 0.010,
}

# Keep requests below org TPM ceilings (large 10-K corpora can exceed limits).
MAX_LLM_CORPUS_CHARS = 18_000


def _compact_corpus(corpus: str, max_chars: int = MAX_LLM_CORPUS_CHARS) -> str:
    """Head+tail truncation with an inline note when a corpus exceeds the cap."""
    if len(corpus) <= max_chars:
        return corpus
    half = max_chars // 2
    head = corpus[:half]
    tail = corpus[-half:]
    note = (
        "\n\n[NOTE: Corpus truncated for API token limits. Preserved beginning and end blocks, "
        f"original_chars={len(corpus)}, kept_chars={len(head) + len(tail)}]\n\n"
    )
    return head + note + tail


def _extract_json_object(text: str) -> dict:
    """Parse a model response into JSON, tolerating code fences/preamble."""
    if text is None:
        raise ValueError("Empty model response (None)")
    t = str(text).strip()
    if not t:
        raise ValueError("Empty model response")

    # Fast path: already valid JSON
    try:
        return json.loads(t)
    except Exception:
        pass

    # Remove markdown fences if present
    if "```" in t:
        t = t.replace("```json", "").replace("```JSON", "").replace("```", "").strip()

    # Extract from first '{' to last '}'
    i = t.find("{")
    j = t.rfind("}")
    if i == -1 or j == -1 or j <= i:
        raise ValueError(f"No JSON object boundaries found. Prefix: {t[:200]!r}")

    candidate = t[i:j+1]
    try:
        return json.loads(candidate)
    except Exception as e:
        raise ValueError(f"Failed to parse extracted JSON: {e}; Prefix: {candidate[:200]!r}")


def _normalize_model_output(obj: dict) -> dict:
    """Enforce the 'criteria' schema key on model output and cached files.

    Self-heals legacy outputs that used the 'pillars' key (from runs before the
    pillars → criteria rename), and raises a clear error if neither key is
    present. Idempotent: a no-op on already-correct payloads.
    """
    if not isinstance(obj, dict):
        raise ValueError("Model output is not a JSON object")
    if "criteria" not in obj and "pillars" in obj:
        obj["criteria"] = obj.pop("pillars")
    if "criteria" in obj and "pillars" in obj:
        obj.pop("pillars", None)
    if "criteria" not in obj or not isinstance(obj["criteria"], dict):
        raise ValueError("Model output missing required 'criteria' object")
    return obj


def score_with_claude(parent: str, corpus: str, max_retries: int = 3) -> tuple[dict, dict]:
    """Returns (parsed_json, usage_dict). Retries on transient errors."""
    if claude_client is None:
        raise RuntimeError("ANTHROPIC_API_KEY not set")
    corpus_llm = _compact_corpus(corpus)
    user_msg = USER_TEMPLATE.format(parent_name=parent, corpus=corpus_llm)
    for attempt in range(max_retries):
        try:
            msg = claude_client.messages.create(
                model=CLAUDE_MODEL,
                max_tokens=2000,
                temperature=0,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": user_msg}],
            )
            text = msg.content[0].text
            parsed = _normalize_model_output(_extract_json_object(text))
            return parsed, {
                "in": msg.usage.input_tokens,
                "out": msg.usage.output_tokens,
                "cost": (msg.usage.input_tokens * COSTS["claude_in"] + msg.usage.output_tokens * COSTS["claude_out"]) / 1000,
            }
        except Exception:
            if attempt == max_retries - 1:
                raise
            time.sleep(2 ** attempt)


def score_with_openai(parent: str, corpus: str, max_retries: int = 3) -> tuple[dict, dict]:
    if openai_client is None:
        raise RuntimeError("OPENAI_API_KEY not set")
    corpus_llm = _compact_corpus(corpus)
    user_msg = USER_TEMPLATE.format(parent_name=parent, corpus=corpus_llm)
    for attempt in range(max_retries):
        try:
            resp = openai_client.chat.completions.create(
                model=OPENAI_MODEL,
                max_tokens=2000,
                temperature=0,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_msg},
                ],
            )
            text = resp.choices[0].message.content
            parsed = _normalize_model_output(_extract_json_object(text))
            return parsed, {
                "in": resp.usage.prompt_tokens,
                "out": resp.usage.completion_tokens,
                "cost": (resp.usage.prompt_tokens * COSTS["openai_in"] + resp.usage.completion_tokens * COSTS["openai_out"]) / 1000,
            }
        except Exception:
            if attempt == max_retries - 1:
                raise
            time.sleep(2 ** attempt)


def score_parent(parent: str, force_rerun: bool = False) -> dict:
    """Score one parent with both APIs. Caches per-(parent, model) results to disk.

    Cached files are normalized on read (legacy 'pillars' key -> 'criteria') and
    rewritten so the on-disk format converges to the current schema.
    """
    corpus = build_corpus(parent)
    if corpus is None:
        return {"parent": parent, "status": "talk_na", "claude": None, "openai": None, "cost": 0.0}

    out_claude = SCRATCH_SCORES / f"{safe_name(parent)}__claude.json"
    out_openai = SCRATCH_SCORES / f"{safe_name(parent)}__openai.json"
    total_cost = 0.0

    if out_claude.exists() and not force_rerun:
        claude_j = _normalize_model_output(json.loads(out_claude.read_text()))
        out_claude.write_text(json.dumps(claude_j, indent=2))
    else:
        claude_j, usage = score_with_claude(parent, corpus)
        out_claude.write_text(json.dumps(claude_j, indent=2))
        total_cost += usage["cost"]

    if out_openai.exists() and not force_rerun:
        openai_j = _normalize_model_output(json.loads(out_openai.read_text()))
        out_openai.write_text(json.dumps(openai_j, indent=2))
    else:
        openai_j, usage = score_with_openai(parent, corpus)
        out_openai.write_text(json.dumps(openai_j, indent=2))
        total_cost += usage["cost"]

    return {"parent": parent, "status": "scored", "claude": claude_j, "openai": openai_j, "cost": total_cost}


In [11]:
# --- One-time cache migration: pillars → criteria ---
# Safety net for runs from before the pillars → criteria rename. Idempotent:
# a no-op on a clean cache or after a previous successful migration.
# Runs at startup so the cache is normalized before the dry-run reads it.
migrated = 0
for fp in SCRATCH_SCORES.glob("*.json"):
    obj = json.loads(fp.read_text())
    if isinstance(obj, dict) and "criteria" not in obj and "pillars" in obj:
        obj["criteria"] = obj.pop("pillars")
        fp.write_text(json.dumps(obj, indent=2))
        migrated += 1
print(f"Migrated cache files: {migrated}")


Migrated cache files: 0


## 4. Dry-run — 5 representative parents

Before paying for a full sweep, score 5 parents chosen to span the *source-mix* combinations the LLMs will face in bulk:

- Two parents with all three sources (10-K + NZT + SBTi) — the richest case.
- One parent with 10-K + NZT only (no SBTi) — the most common case in this cohort.
- One parent whose NZT entry says "No target" — a hard case that should pull criterion scores down.
- One recently-IPO'd parent — to surface any spin-off-specific reporting gaps.

The dry-run prints per-criterion Claude vs. GPT-4o scores side-by-side and an aggregate inter-model agreement summary (exact match / within-1 / within-2). If a reviewer wants to iterate on the rubric or system prompt at this point, re-running with `force_rerun=True` bypasses the disk cache for these 5 parents only.

In [12]:
DRY_RUN = [
    "DUKE ENERGY CORP",            # strong commitment + 10-K + NZT
    "Vistra Corp",                 # SBTi + NZT + 10-K (triple source)
    "NRG ENERGY INC",              # SBTi-removed + NZT + 10-K
    "CONSTELLATION ENERGY CORP",   # recent Exelon spin-off + NZT + 10-K
    "PINNACLE WEST CAPITAL CORP",  # NZT says "No target" + 10-K (weak case)
]

dry_results = []
total_cost = 0.0
for parent in DRY_RUN:
    print(f"\\nScoring {parent} ...")
    r = score_parent(parent)
    dry_results.append(r)
    total_cost += r["cost"]
    if r["status"] == "scored":
        cs = {p: r["claude"]["criteria"][p]["score"] for p in ["ambition", "specificity", "time_horizon", "mechanism_clarity", "verification_governance"]}
        os_ = {p: r["openai"]["criteria"][p]["score"] for p in ["ambition", "specificity", "time_horizon", "mechanism_clarity", "verification_governance"]}
        print(f"  CLAUDE: {cs}  conf={r['claude'].get('confidence')}")
        print(f"  OPENAI: {os_}  conf={r['openai'].get('confidence')}")
    else:
        print(f"  STATUS: {r['status']}")
print(f"\\nDry-run total cost: ${total_cost:.4f}")

\nScoring DUKE ENERGY CORP ...
  CLAUDE: {'ambition': 6, 'specificity': 6, 'time_horizon': 6, 'mechanism_clarity': 4, 'verification_governance': 6}  conf=medium
  OPENAI: {'ambition': 10, 'specificity': 6, 'time_horizon': 6, 'mechanism_clarity': 2, 'verification_governance': 4}  conf=medium
\nScoring Vistra Corp ...
  CLAUDE: {'ambition': 8, 'specificity': 7, 'time_horizon': 6, 'mechanism_clarity': 3, 'verification_governance': 5}  conf=medium
  OPENAI: {'ambition': 8, 'specificity': 8, 'time_horizon': 8, 'mechanism_clarity': 2, 'verification_governance': 4}  conf=medium
\nScoring NRG ENERGY INC ...
  CLAUDE: {'ambition': 8, 'specificity': 7, 'time_horizon': 6, 'mechanism_clarity': 2, 'verification_governance': 5}  conf=medium
  OPENAI: {'ambition': 10, 'specificity': 6, 'time_horizon': 6, 'mechanism_clarity': 4, 'verification_governance': 4}  conf=medium
\nScoring CONSTELLATION ENERGY CORP ...
  CLAUDE: {'ambition': 6, 'specificity': 6, 'time_horizon': 6, 'mechanism_clarity': 2, 'veri

In [13]:
# --- Side-by-side comparison table ---
rows = []
for r in dry_results:
    if r["status"] != "scored": continue
    for criterion in ["ambition", "specificity", "time_horizon", "mechanism_clarity", "verification_governance"]:
        rows.append({
            "parent": r["parent"], "criterion": criterion,
            "claude": r["claude"]["criteria"][criterion]["score"],
            "openai": r["openai"]["criteria"][criterion]["score"],
        })
cmp_df = pd.DataFrame(rows)
cmp_df["diff"] = cmp_df["claude"] - cmp_df["openai"]
cmp_df["abs_diff"] = cmp_df["diff"].abs()

print("Side-by-side scores:")
print(cmp_df.to_string(index=False))

# Inter-model agreement
print("\nInter-model agreement per criterion:")
for criterion in ["ambition", "specificity", "time_horizon", "mechanism_clarity", "verification_governance"]:
    sub = cmp_df[cmp_df["criterion"] == criterion].dropna(subset=["claude", "openai"])
    if len(sub) >= 3:
        r = sub["claude"].corr(sub["openai"])
        mad = sub["abs_diff"].mean()
        print(f"  {criterion:25s}: Pearson r = {r:+.3f}  MAD = {mad:.2f} points  (n={len(sub)})")

# Aggregate: how often do models agree exactly? Within 1? Within 2?
exact = (cmp_df["abs_diff"] == 0).mean()
within_1 = (cmp_df["abs_diff"] <= 1).mean()
within_2 = (cmp_df["abs_diff"] <= 2).mean()
print(f"\nAggregate agreement (n={len(cmp_df)} criterion-parent observations):")
print(f"  Exact match:  {exact:.1%}")
print(f"  Within 1 pt:  {within_1:.1%}")
print(f"  Within 2 pts: {within_2:.1%}")

Side-by-side scores:
                    parent               criterion  claude  openai  diff  abs_diff
          DUKE ENERGY CORP                ambition       6      10    -4         4
          DUKE ENERGY CORP             specificity       6       6     0         0
          DUKE ENERGY CORP            time_horizon       6       6     0         0
          DUKE ENERGY CORP       mechanism_clarity       4       2     2         2
          DUKE ENERGY CORP verification_governance       6       4     2         2
               Vistra Corp                ambition       8       8     0         0
               Vistra Corp             specificity       7       8    -1         1
               Vistra Corp            time_horizon       6       8    -2         2
               Vistra Corp       mechanism_clarity       3       2     1         1
               Vistra Corp verification_governance       5       4     1         1
            NRG ENERGY INC                ambition       8      10

## Interpreting Cell 24 Output

Cell 24 reports inter-model agreement between Claude and GPT-4o on the dry-run criterion scores.

- Pearson r (per criterion): captures directional consistency across parents.
- r near +1.0 means both models rank severity similarly for that criterion.
- r near 0 means weak linear agreement.
- Negative r means inverse scoring patterns and likely disagreement.

- MAD (Mean Absolute Difference): average score gap between models on the 0-10 scale.
- Lower MAD is better.
- MAD = 0 means exact agreement.
- Rough guide: <=1 tight agreement, 1-2 moderate, >2 notable divergence.

Aggregate agreement metrics:
- Exact match: share of criterion-parent pairs with identical model scores.
- Within 1 point: share where absolute difference is <=1.
- Within 2 points: share where absolute difference is <=2.

Interpretation rule of thumb:
- High Pearson r together with low MAD indicates strong rubric consistency across models.
- Low r with high MAD flags criteria where rubric wording or evidence extraction may need refinement before relying on bulk scores.

## 5. Bulk scoring — all remaining parents

This cell scores every cohort parent that has a disclosure corpus on both APIs. It is idempotent: the parents already scored in Section 4 are skipped automatically because `score_parent()` reads cached per-(parent, model) JSON from disk. To force a clean re-run, delete files in `data/processed/scratch/llm_scores/` before executing.

When this cell finishes, every cohort parent with a corpus has two JSON score files on disk (one per model), each containing criterion scores + reasoning + evidence quotes + supplemental indicators. Parents flagged Talk-NA in Section 1 are correctly skipped (status `"talk_na"`).

In [14]:
# Score every parent with disclosure (already-scored cached on disk)
all_results = []
bulk_cost = 0.0
parents_to_score = corpus_stats[corpus_stats["has_corpus"]]["parent"].tolist()
print(f"Parents to score: {len(parents_to_score)} (skipping {(corpus_stats['has_corpus'] == False).sum()} Talk-NA parents)")

for parent in tqdm(parents_to_score, desc="Scoring"):
    r = score_parent(parent)
    all_results.append(r)
    bulk_cost += r["cost"]

print(f"\\nBulk total NEW cost (not counting cached): ${bulk_cost:.4f}")
print(f"Total parents scored: {sum(1 for r in all_results if r['status'] == 'scored')}")

Parents to score: 26 (skipping 24 Talk-NA parents)


Scoring: 100%|██████████| 26/26 [00:00<00:00, 1419.84it/s]

\nBulk total NEW cost (not counting cached): $0.0000
Total parents scored: 26


## 6. Composite PQS

For each parent and each criterion, the composite uses the **mean of Claude and GPT-4o scores** when both are available, falling back to the single model when one is missing/null. This is the most defensible single-number per-criterion aggregation given two independent raters of similar capability: it halves the variance of each individual model's noise while not privileging either rater.

**Criterion weighting.** The composite is the **equal-weighted mean of the 5 criterion means**, then scaled ×10 to land on a 0–100 scale. Equal weights are the conservative default in the absence of an external mandate ranking the criteria (Section 7 tests rank stability under three deliberately skewed weight schemes). Notebook 07 will additionally expose interactive weight sliders so stakeholders can re-rank parents under their own priorities.

**SBTi bonus.** Parents whose `sbti_status` is `validated_targets` or `targets_set` receive a **+5 point bonus** (composite capped at 100). The rationale is that SBTi validation is an external, binary credibility signal — the company has subjected its target to independent technical review against a published methodology — which the text-based criterion scoring cannot directly capture. The +5 magnitude is small enough that it cannot flip a low-criterion parent into the top quartile, but large enough to meaningfully separate two parents with otherwise comparable criterion text. `removed` and `committed` statuses do **not** receive the bonus (removed = SBTi rejected the target; committed = pledge without validated target).

Talk-NA parents (no disclosure source) receive `pqs_composite = NaN` and are flagged for Notebook 06 to handle explicitly.

In [15]:
# Assemble per-parent per-criterion scores into a dataframe
pqs_rows = []
for r in all_results:
    if r["status"] != "scored":
        # Talk-NA
        pqs_rows.append({"parent_name": r["parent"], "status": "talk_na",
                          "pqs_composite": np.nan, "sbti_status": None})
        continue
    row = {"parent_name": r["parent"], "status": "scored"}
    for criterion in ["ambition", "specificity", "time_horizon", "mechanism_clarity", "verification_governance"]:
        c = r["claude"]["criteria"][criterion].get("score")
        o = r["openai"]["criteria"][criterion].get("score")
        # Average available models
        vals = [v for v in (c, o) if v is not None]
        row[f"{criterion}_claude"] = c
        row[f"{criterion}_openai"] = o
        row[f"{criterion}_mean"]   = float(np.mean(vals)) if vals else np.nan
    # SBTi status (use Claude's; fallback OpenAI)
    sbti_c = r["claude"].get("supplemental_indicators", {}).get("sbti_status")
    sbti_o = r["openai"].get("supplemental_indicators", {}).get("sbti_status")
    row["sbti_status"] = sbti_c or sbti_o
    pqs_rows.append(row)

pqs = pd.DataFrame(pqs_rows)

# Composite: equal-weighted mean of 5 criterion means * 10 → 0-100 scale
criterion_mean_cols = [f"{p}_mean" for p in ["ambition","specificity","time_horizon","mechanism_clarity","verification_governance"]]
pqs["pqs_raw_5criterion"] = pqs[criterion_mean_cols].mean(axis=1) * 10  # 0-10 scale × 10 = 0-100

# SBTi bonus
def sbti_bonus(s):
    if s in ("validated_targets", "targets_set"): return 5.0
    return 0.0
pqs["sbti_bonus"] = pqs["sbti_status"].map(sbti_bonus).fillna(0)
pqs["pqs_composite"] = (pqs["pqs_raw_5criterion"] + pqs["sbti_bonus"]).clip(upper=100)

print(f"Parents scored: {pqs['status'].eq('scored').sum()}")
print(f"Talk-NA:        {pqs['status'].eq('talk_na').sum()}")
print(f"\nTop 10 by PQS composite:")
print(pqs[pqs['status']=='scored'].nlargest(10, 'pqs_composite')[['parent_name','pqs_raw_5criterion','sbti_bonus','pqs_composite','sbti_status']].to_string(index=False))
print(f"\nBottom 10 by PQS composite:")
print(pqs[pqs['status']=='scored'].nsmallest(10, 'pqs_composite')[['parent_name','pqs_raw_5criterion','sbti_bonus','pqs_composite','sbti_status']].to_string(index=False))

Parents scored: 26
Talk-NA:        0

Top 10 by PQS composite:
              parent_name  pqs_raw_5criterion  sbti_bonus  pqs_composite sbti_status
       NEXTERA ENERGY INC                74.0         0.0           74.0 not_present
            DTE ENERGY CO                73.0         0.0           73.0 not_present
              AMEREN CORP                66.0         0.0           66.0     unknown
          XCEL ENERGY INC                66.0         0.0           66.0 not_present
              Vistra Corp                59.0         5.0           64.0 targets_set
           NRG ENERGY INC                58.0         5.0           63.0 targets_set
         EXXON MOBIL CORP                58.0         0.0           58.0 not_present
CONSTELLATION ENERGY CORP                56.0         0.0           56.0 not_present
         DUKE ENERGY CORP                56.0         0.0           56.0 not_present
  MARATHON PETROLEUM CORP                54.0         0.0           54.0 not_present

B

## 7. Sensitivity analysis on criterion weights

Section 6 uses equal weights (0.2 each). This section tests how sensitive the per-parent ranking is to that choice by re-computing the composite under three deliberately-skewed weight schemes and reporting the Spearman rank correlation across all four schemes.

Why this matters: if the ranking is robust (Spearman ρ > 0.9 across schemes), the equal-weight default can be defended as informative regardless of stakeholder priorities. If the ranking is weight-dependent (ρ < 0.7), the headline composite must be presented alongside its weighting choice — and the interactive weight sliders in Notebook 07's dashboard become essential context rather than a convenience.

Schemes tested: equal; ambition-heavy (0.4 on Ambition, 0.15 on the others); mechanism-heavy (0.4 on Mechanism Clarity); verification-heavy (0.4 on Verification & Governance).

In [16]:
CRITERION_KEYS = ["ambition", "specificity", "time_horizon", "mechanism_clarity", "verification_governance"]

WEIGHT_SCHEMES = {
    "equal":                  {"ambition": 0.2, "specificity": 0.2, "time_horizon": 0.2, "mechanism_clarity": 0.2, "verification_governance": 0.2},
    "ambition_heavy":         {"ambition": 0.4, "specificity": 0.15, "time_horizon": 0.15, "mechanism_clarity": 0.15, "verification_governance": 0.15},
    "mechanism_heavy":        {"ambition": 0.15, "specificity": 0.15, "time_horizon": 0.15, "mechanism_clarity": 0.40, "verification_governance": 0.15},
    "verification_heavy":     {"ambition": 0.15, "specificity": 0.15, "time_horizon": 0.15, "mechanism_clarity": 0.15, "verification_governance": 0.40},
}

sens_rows = []
scored = pqs[pqs["status"] == "scored"].copy()
for name, w in WEIGHT_SCHEMES.items():
    scores = sum(scored[f"{p}_mean"] * w[p] for p in CRITERION_KEYS) * 10
    sens_rows.append((name, scores))

sens_df = scored[["parent_name"]].copy()
for name, scores in sens_rows:
    sens_df[name] = scores.values
sens_df.set_index("parent_name", inplace=True)

# Rank stability check: how much do ranks change?
rank_df = sens_df.rank(ascending=False)
rank_corr = rank_df.corr(method="spearman")
print("Spearman rank correlation across weight schemes:")
print(rank_corr.round(3).to_string())
print(f"\\nMin Spearman: {rank_corr.values[rank_corr.values < 1.0].min():.3f}")
print(f"  (>0.9 = rankings are robust to weight choice; <0.7 = ranking is weight-dependent)")

Spearman rank correlation across weight schemes:
                    equal  ambition_heavy  mechanism_heavy  verification_heavy
equal               1.000           0.975            0.970               0.981
ambition_heavy      0.975           1.000            0.949               0.949
mechanism_heavy     0.970           0.949            1.000               0.940
verification_heavy  0.981           0.949            0.940               1.000
\nMin Spearman: 0.940
  (>0.9 = rankings are robust to weight choice; <0.7 = ranking is weight-dependent)


## 8. Save deliverables

In [17]:
# --- Main deliverable: cohort_pqs.csv ---
out_cols = ["parent_name", "status", "pqs_composite", "pqs_raw_5criterion", "sbti_status", "sbti_bonus"]
out_cols += [f"{p}_mean"   for p in CRITERION_KEYS]
out_cols += [f"{p}_claude" for p in CRITERION_KEYS]
out_cols += [f"{p}_openai" for p in CRITERION_KEYS]
pqs[out_cols].to_csv(DATA_PROCESSED / "cohort_pqs.csv", index=False)
print(f"Wrote cohort_pqs.csv: {len(pqs)} parents × {len(out_cols)} cols")

# --- Inter-model agreement table ---
agreement_rows = []
scored_r = [r for r in all_results if r["status"] == "scored"]
for criterion in CRITERION_KEYS:
    cv = [r["claude"]["criteria"][criterion].get("score") for r in scored_r]
    ov = [r["openai"]["criteria"][criterion].get("score") for r in scored_r]
    df_ = pd.DataFrame({"c": cv, "o": ov}).dropna()
    if len(df_) >= 5:
        agreement_rows.append({
            "criterion": criterion,
            "n": len(df_),
            "pearson_r": df_["c"].corr(df_["o"]),
            "mean_abs_diff": (df_["c"] - df_["o"]).abs().mean(),
            "exact_match_pct": (df_["c"] == df_["o"]).mean() * 100,
            "within_1pt_pct":  ((df_["c"] - df_["o"]).abs() <= 1).mean() * 100,
        })
agreement_df = pd.DataFrame(agreement_rows)
agreement_df.to_csv(DATA_PROCESSED / "cohort_pqs_intermodel_agreement.csv", index=False)
print(f"Wrote cohort_pqs_intermodel_agreement.csv")
print(agreement_df.to_string(index=False))

# --- Reasoning JSON ---
reasoning = {r["parent"]: {"claude": r["claude"], "openai": r["openai"]}
              for r in scored_r}
with open(DATA_PROCESSED / "cohort_pqs_llm_reasoning.json", "w") as f:
    json.dump(reasoning, f, indent=2)
print(f"Wrote cohort_pqs_llm_reasoning.json ({len(reasoning)} parents)")

Wrote cohort_pqs.csv: 26 parents × 21 cols
Wrote cohort_pqs_intermodel_agreement.csv
              criterion  n  pearson_r  mean_abs_diff  exact_match_pct  within_1pt_pct
               ambition 26   0.886810       2.076923        30.769231       34.615385
            specificity 26   0.916126       1.000000        50.000000       65.384615
           time_horizon 26   0.969997       0.615385        65.384615       73.076923
      mechanism_clarity 26   0.616902       1.423077        34.615385       46.153846
verification_governance 25   0.818560       0.840000        48.000000       68.000000
Wrote cohort_pqs_llm_reasoning.json (26 parents)


## 9. Summary of Results

### 9.1 What is delivered

| Quantity | Value |
|---|---|
| Cohort parents | 50 |
| Parents with PQS score | (see printout below) |
| Talk-NA parents (no disclosure) | (see printout below) |
| LLM raters | Claude Sonnet 4.6 + GPT-4o, both at `temperature=0` |
| Total API cost | (printed by Sections 4 and 5) |
| Inter-model agreement | (saved as `cohort_pqs_intermodel_agreement.csv`) |

### 9.2 Methodological transparency

- **SBTi coverage gap is itself a finding.** Only 2 of 50 cohort parents have SBTi targets (Section 1 coverage probe). This is the empirical justification for broadening the disclosure source set rather than a data-acquisition shortfall.
- **Multi-source triangulation** (SBTi + Net Zero Tracker + SEC 10-K) brings Talk-axis coverage to 26 of 50 parents. The remaining 24 are cooperatives, public-power authorities, PE-owned holdcos, and government entities for which no public disclosure source we can access exists.
- **Dual-LLM scoring** replaces regex/keyword-density approaches that would conflate hedged with concrete language. Inter-model agreement is reported per-criterion as a reproducibility check.
- **Composite construction** uses equal-weighted 5-criterion mean × 10 = 0–100, plus a +5 SBTi-validation bonus capped at 100 (rationale in Section 6).
- **Audit trail.** Every score carries a 1–2 sentence reasoning and a verbatim quote from the disclosure text, persisted in `cohort_pqs_llm_reasoning.json` for downstream review.

### 9.3 Limitations (what this notebook does NOT do)

- We do not search beyond SEC 10-K, NZT, and SBTi. Company sustainability reports, CDP responses, and proxy statements would add coverage but require additional acquisition work outside this notebook's scope.
- We do not down-weight LLM scores by the models' self-reported `confidence` field, although the field is captured in the saved reasoning JSON and is available for a future-work enhancement.
- The 24 Talk-NA parents inherit `pqs_composite = NaN`. Notebook 06 must decide whether to score them as a partial composite (Walk × Verify only) or carry them with NaN through to the final CCCS.

### 9.4 Next: Notebook 06

Notebook 06 combines `cohort_pqs.csv` (Talk) with the Walk-axis (decarbonization velocity from Notebook 01 + virtual baseline from Notebook 04) and the Verify-axis (SCVS from Notebook 03's LME residuals) into the headline Corporate Climate Credibility Score (CCCS) per parent and the 2×2 quadrant assignment.

In [18]:
# --- Programmatic summary printout ---
print("=" * 72)
print("NOTEBOOK 05 — SUMMARY OF RESULTS")
print("=" * 72)
print(f"\\nCohort: {len(pqs)} parents")
print(f"  Scored:   {pqs['status'].eq('scored').sum()}")
print(f"  Talk-NA:  {pqs['status'].eq('talk_na').sum()}")
print(f"\\nPQS composite (scored parents):")
print(f"  mean   = {pqs[pqs['status']=='scored']['pqs_composite'].mean():.1f}")
print(f"  median = {pqs[pqs['status']=='scored']['pqs_composite'].median():.1f}")
print(f"  range  = [{pqs[pqs['status']=='scored']['pqs_composite'].min():.1f}, {pqs[pqs['status']=='scored']['pqs_composite'].max():.1f}]")
print(f"\\nInter-model agreement summary:")
print(f"  Mean Pearson r across 5 criteria: {agreement_df['pearson_r'].mean():.3f}")
print(f"  Mean abs-diff across 5 criteria:  {agreement_df['mean_abs_diff'].mean():.2f} points (on 0-10 scale)")
print("=" * 72)

NOTEBOOK 05 — SUMMARY OF RESULTS
\nCohort: 26 parents
  Scored:   26
  Talk-NA:  0
\nPQS composite (scored parents):
  mean   = 42.6
  median = 49.5
  range  = [0.0, 74.0]
\nInter-model agreement summary:
  Mean Pearson r across 5 criteria: 0.842
  Mean abs-diff across 5 criteria:  1.19 points (on 0-10 scale)
